# MedBridge AI — HF Loading Test: Omni Arabic ASR

This notebook verifies that the **Arabic source dataset** and the **Omni Arabic ASR model** can be loaded from Hugging Face.

Important clarification:

- The Arabic ASR data used by MedBridge AI is derived from the public dataset: `ArabicSpeech/MGB-3`
- The Arabic Omni model repository is: `medbridge-ai/omni-ar-asr`

This notebook validates:

1. Access to the source dataset `ArabicSpeech/MGB-3`
2. Basic inspection of dataset metadata / sample structure
3. Access to the model `medbridge-ai/omni-ar-asr`
4. Presence of the checkpoint and metadata files


In [ ]:
!pip install -U huggingface_hub datasets pyarrow pandas soundfile


## 1. Configuration


In [ ]:
import os
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HOME"] = "/content/hf_cache_medbridge"

# Source dataset used for Egyptian Arabic ASR
SOURCE_DATASET_REPO = "ArabicSpeech/MGB-3"

# MedBridge model repository
MODEL_REPO = "medbridge-ai/omni-ar-asr"

print("HF_HOME:", os.environ["HF_HOME"])
print("Source dataset repo:", SOURCE_DATASET_REPO)
print("Model repo:", MODEL_REPO)


## 2. Inspect the source dataset: ArabicSpeech/MGB-3

The original data source for the Arabic ASR component is the **MGB-3 Egyptian Arabic speech dataset**.

We use `streaming=True` to avoid downloading the full dataset in Colab.


In [ ]:
from datasets import get_dataset_config_names, get_dataset_split_names

try:
    configs = get_dataset_config_names(SOURCE_DATASET_REPO, trust_remote_code=True)
except Exception as e:
    configs = []
    print("Could not list configs:", repr(e))

print("Available configs:", configs if configs else "No explicit configs detected / not required")

# Try to inspect splits. Some datasets require a config name; this block handles both cases.
try:
    splits = get_dataset_split_names(SOURCE_DATASET_REPO, trust_remote_code=True)
    selected_config = None
except Exception as e:
    print("Could not list splits without config:", repr(e))
    selected_config = configs[0] if configs else None
    if selected_config:
        splits = get_dataset_split_names(SOURCE_DATASET_REPO, selected_config, trust_remote_code=True)
    else:
        splits = []

print("Selected config:", selected_config)
print("Available splits:", splits)


In [ ]:
from datasets import load_dataset

# Pick a split safely.
preferred_splits = ["train", "validation", "dev", "test"]
selected_split = next((s for s in preferred_splits if s in splits), None)

if selected_split is None:
    # Fallback: many HF datasets expose at least one split.
    selected_split = splits[0] if splits else "train"

print("Selected split:", selected_split)

# Load only in streaming mode to avoid downloading the full dataset.
if selected_config:
    ds_stream = load_dataset(
        SOURCE_DATASET_REPO,
        selected_config,
        split=selected_split,
        streaming=True,
        trust_remote_code=True,
    )
else:
    ds_stream = load_dataset(
        SOURCE_DATASET_REPO,
        split=selected_split,
        streaming=True,
        trust_remote_code=True,
    )

print(ds_stream)


In [ ]:
# Read one sample to verify accessibility and inspect the schema.
sample = next(iter(ds_stream))

print("Sample keys:", list(sample.keys()))

for key, value in sample.items():
    if key.lower() in ["audio", "speech", "array"]:
        print(f"{key}: audio-like field ->", type(value))
    else:
        text_value = str(value)
        print(f"{key}:", text_value[:300])


## 3. Optional: inspect the MedBridge converted dataset repository

If the converted Omni-format dataset has also been uploaded to `medbridge-ai/asr-ar-omni`, this optional section checks its parquet structure.

If the repo only contains documentation or has not yet been fully uploaded, this section may report zero parquet files. That is not a failure for the source dataset test above.


In [ ]:
from huggingface_hub import snapshot_download

MEDBRIDGE_DATASET_REPO = "medbridge-ai/asr-ar-omni"

try:
    medbridge_dataset_dir = snapshot_download(
        repo_id=MEDBRIDGE_DATASET_REPO,
        repo_type="dataset",
    )
    medbridge_dataset_dir = Path(medbridge_dataset_dir)
    print("MedBridge dataset directory:", medbridge_dataset_dir)
    print("Top-level files/folders:")
    for item in medbridge_dataset_dir.iterdir():
        print("-", item.name)

    parquets = list(medbridge_dataset_dir.rglob("*.parquet"))
    print("Number of parquet files:", len(parquets))
    for p in parquets[:5]:
        print("-", p.relative_to(medbridge_dataset_dir))
except Exception as e:
    print("Optional MedBridge dataset inspection skipped or failed:")
    print(repr(e))


## 4. Download and inspect the Omni Arabic model


In [ ]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(
    repo_id=MODEL_REPO,
    repo_type="model",
)

model_dir = Path(model_dir)
print("Model directory:", model_dir)
print("Top-level files/folders:")
for item in model_dir.iterdir():
    print("-", item.name)


In [ ]:
# Expected files may vary depending on how the Arabic model was exported.
# Current expected layout may include either:
# 1) best_model.pt at root
# 2) model/pp_00/tp_00/sdp_00.pt + YAML config files

possible_checkpoint_paths = [
    model_dir / "best_model.pt",
    model_dir / "model" / "pp_00" / "tp_00" / "sdp_00.pt",
]

print("Checkpoint candidates:")
found_ckpt = None
for path in possible_checkpoint_paths:
    status = "OK" if path.exists() else "MISSING"
    print(f"- {path.relative_to(model_dir)}: {status}")
    if path.exists():
        found_ckpt = path

assert found_ckpt is not None, "No checkpoint found. Expected best_model.pt or model/pp_00/tp_00/sdp_00.pt."

print("Selected checkpoint:", found_ckpt.relative_to(model_dir))
print("Checkpoint size GB:", round(found_ckpt.stat().st_size / 1e9, 3))


In [ ]:
expected_metadata = [
    "README.md",
    "asset_card.yaml",
    "model.yaml",
    "train_config.yaml",
]

print("Metadata/config files:")
for f in expected_metadata:
    path = model_dir / f
    status = "OK" if path.exists() else "MISSING"
    print(f"- {f}: {status}")


## 5. Final validation


In [ ]:
assert sample is not None, "Dataset validation failed: no sample could be read from ArabicSpeech/MGB-3."
assert found_ckpt.exists(), "Model validation failed: checkpoint not found."

print("✅ Source dataset ArabicSpeech/MGB-3 and model medbridge-ai/omni-ar-asr are accessible from Hugging Face.")


## Notes

This notebook validates **repository accessibility and file integrity**.

It does not run full OmniASR inference because the Omni inference stack may require the project-specific environment (`omnilingual_asr`, `fairseq2`, CUDA setup, and compatible audio dependencies).
